In [1]:
from openai import OpenAI
import openai
import pandas as pd
import requests

In [2]:
client = OpenAI(
    api_key="token-vulcan",  # 输入你的 API Key
    base_url="http://219.147.99.170:40019/v1"
)




def qa_base(messages):
    completion = client.chat.completions.create(
        model="Qwen2-5-14B",
        messages=messages,
        logprobs=False,

        # stream=True  # 开启流式输出,
    )
    return completion.choices[0].message.content
    # if compile.status_code == 200:
    #     result = completion[0].choices[0].delta.content
    #     return result
    # else:
    #     print(f"请求失败，状态码: {completion.status_code}")
    #     return ""

user_input = "中国的首都是哪里"
input = [{"role": "user", "content": user_input},]
qa_base(input)

'中国的首都是北京。'

# 输出结构

In [36]:
output_struct_ct = {
    "报告信息":["report_id","patient_id","exam_date","exam_location","肺部结构是否正常"],
    "肺部结构信息":["layer_structure","airway_description","mediastinal_findings","cardiac_large_vessels","pleural_and_cavity_findings"],
    "左侧肺部信息":["side","nodules_present","nodules_count"],
    "左侧肺部结节 1":["nodule_id","location","size_long_axis_cm","size_short_axis_cm","density","shape","suspected_diagnosis"],
    "左侧肺部结节 2":["nodule_id","location","size_long_axis_cm","size_short_axis_cm","density","shape","suspected_diagnosis"],
    "左侧肺部低密度区 1":["low_density_id","location","size_long_axis_cm","size_width_axis_cm","size_height_axis_cm","shape","suspected_diagnosis"],
    "右侧肺部信息":["side","nodules_present","nodules_count"],
    "右侧肺部结节 1":["nodule_id","location","size_long_axis_cm","size_short_axis_cm","density","shape","suspected_diagnosis"],
    "右侧肺部结节 2":["nodule_id","location","size_long_axis_cm","size_short_axis_cm","density","shape","suspected_diagnosis"],
    "右侧肺部结节 3":["nodule_id","location","size_long_axis_cm","size_short_axis_cm","density","shape","suspected_diagnosis"],
    "结论":["conclusion"]
}

output_struct_rx = {
    "报告信息":["report_id","patient_id","exam_date","layer_structure","gland_arrangement","axillary_findings","supraclavicular_findings","CDFI_findings"],
    "左侧乳腺信息":["side","duct_dilation","duct_dilation_details","blood_flow_present","blood_flow_details","nodules_present","nodules_count"],
    "左侧乳腺结节 1":["nodule_id","location","distance_from_nipple_cm","size_x_cm","size_y_cm","boundary","shape","blood_flow","blood_flow_details"],
    "左侧乳腺结节 2":["nodule_id","location","distance_from_nipple_cm","size_x_cm","size_y_cm","boundary","shape","blood_flow","blood_flow_details"],
    "右侧乳腺信息":["side","duct_dilation","duct_dilation_details","blood_flow_present","blood_flow_details","nodules_present","nodules_count"],
    "右侧乳腺结节 1":["nodule_id","location","distance_from_nipple_cm","size_x_cm","size_y_cm","boundary","shape","blood_flow","blood_flow_details"],
    "右侧乳腺结节 2":["nodule_id","location","distance_from_nipple_cm","size_x_cm","size_y_cm","boundary","shape","blood_flow","blood_flow_details"],
    "右侧乳腺结节 3":["nodule_id","location","distance_from_nipple_cm","size_x_cm","size_y_cm","boundary","shape","blood_flow","blood_flow_details"],
    "结论":["conclusion"]
}

In [41]:
output_struct_ct_dict = {}
for key,values in output_struct_ct.items():
    # print(key)
    # print(values)
    output_struct_ct_dict[key] = {}
    for v in values:
        output_struct_ct_dict[key][v] = ""
# output_struct_ct_dict

output_struct_rx_dict = {}
for key,values in output_struct_rx.items():
    # print(key)
    # print(values)
    output_struct_rx_dict[key] = {}
    for v in values:
        output_struct_rx_dict[key][v] = ""

## 乳腺

In [68]:
rx_output = {'报告信息': {'report_id': '1',
  'patient_id': '1001',
  'exam_date': '2024-12-17',
  'layer_structure': '层次结构不清晰',
  'gland_arrangement': '腺体排列异常',
  'axillary_findings': '双侧腋窝及锁骨下未见明显肿大的淋巴结回声',
  'supraclavicular_findings': '双侧锁骨上未见明显异常占位回声',
  'CDFI_findings': '未见明显血流信号'},
 '左侧乳腺信息': {'side': '左',
  'duct_dilation': 'FALSE',
  'duct_dilation_details': '',
  'blood_flow_present': 'FALSE',
  'blood_flow_details': '未见明显血流信号',
  'nodules_present': 'TRUE',
  'nodules_count': '1'},
 '左侧乳腺结节 1': {'nodule_id': '1',
  'location': '9 点方向',
  'distance_from_nipple_cm': 'NULL',
  'size_x_cm': '0.75',
  'size_y_cm': '0.43',
  'boundary': '清',
  'shape': '规则',
  'blood_flow': 'FALSE',
  'blood_flow_details': '未见明显血流信号'},
 '右侧乳腺信息': {'side': '右',
  'duct_dilation': 'FALSE',
  'duct_dilation_details': '',
  'blood_flow_present': 'FALSE',
  'blood_flow_details': '未见明显血流信号',
  'nodules_present': 'FALSE',
  'nodules_count': '0'},
 '结论': {'conclusion': '左乳腺结节 BI-RADS3 类；右侧乳腺未见明显占位病变；双侧腋窝淋巴结'}}


In [81]:
prompt_rx = """
任务说明：

您是一个医疗数据提取助手，专门用于解析乳腺超声报告。您的任务是从给定的报告文本中提取以下结构化信息，并将其填写在指定的单一垂直表格中。请确保信息的准确性和完整性，特别是在提取结节描述和结论部分。

需要提取的信息类别：
1.乳腺结构正常性：
•乳腺层次结构是否清晰,相关字段:layer_structure。
•腺体排列是否正常,相关字段:gland_arrangement。
2.乳腺结节的结构化描述：
•结节所在侧别（左或右）。
•结节位置（如“6点钟”）,相关字段:location。
•结节大小（长轴和短轴，单位为厘米）,相关字段:size_x_cm,size_y_cm。
•结节边界描述（如“尚清”）,相关字段:boundary。
•结节形态描述（如“规则”）,相关字段:shape。
•是否见明显血流信号,相关字段:boundary。
•结节的疑似诊断（如“纤维腺瘤？”）。
3.乳腺低回声区的描述：
•低回声区所在侧别。
•低回声区位置。
•低回声区大小（长轴、宽轴、高轴，单位为厘米）。
•低回声区边界描述。
•低回声区形态描述。
•是否见明显血流信号。
•低回声区的疑似诊断（如“腺病？”）。
4.报告结论的结构化描述(相关字段:conclusion):
•总结报告中的结论部分，示例('报告结论':'左乳腺结节BI-RADS3类；右侧乳腺未见明显占位病变；双侧腋窝淋巴结' ‘结构化描述’:'左乳腺结节；右侧乳腺未见明显占位病变：双侧腋窝淋巴结')

表格格式要求：

将所有提取的信息按照以下垂直排列的表格格式填写。每个字段在表格的第一列，相关值在第二列,字段和相关值用#连接。对于存在多个结节或低回声区的情况，使用编号区分每个结节或低回声区的信息。

字段值
报告信息
report_id[填写报告唯一标识，例如1]
patient_id[填写患者唯一标识，例如1001]
exam_date[填写检查日期，例如2024-12-17]
layer_structure[填写乳腺层次结构描述，例如“层次结构清晰”或“层次结构不清”]
gland_arrangement[填写腺体排列描述，例如“腺体排列尚可”或“腺体排列异常”]
axillary_findings[填写双侧腋窝及锁骨下淋巴结探查结果，例如“双侧腋窝及锁骨下未见明显肿大的淋巴结回声”]
supraclavicular_findings[填写双侧锁骨上淋巴结探查结果，例如“双侧锁骨上未见明显异常占位回声”]
CDFI_findings[填写彩色多普勒血流成像结果，例如“未见明显血流信号”或“见异常血流信号”]
乳腺结构是否正常[填写“正常”或“不正常”，根据layer_structure和gland_arrangement的描述]
左侧乳腺信息
side左
duct_dilation[填写TRUE/FALSE，是否有导管扩张，根据报告内容]
duct_dilation_details[填写导管扩张的具体描述，如“局限性增宽，内径约0.23cm”或留空]
blood_flow_present[填写TRUE/FALSE，是否存在异常血流信号]
blood_flow_details[填写血流信号的具体描述，如“未见明显血流信号”或具体血流描述]
nodules_present[填写TRUE/FALSE，是否存在结节]
nodules_count[填写结节数量，如2]
左侧乳腺结节 1
nodule_id1
location6点钟
distance_from_nipple_cm[填写距乳头距离，若报告中未提及则填写NULL]
size_x_cm1.2
size_y_cm0.6
boundary尚清
shape规则
blood_flowFALSE
blood_flow_details未见明显血流信号
左侧乳腺结节 2
nodule_id2
location纤维腺瘤（疑）
distance_from_nipple_cmNULL
size_x_cm1.2
size_y_cm0.6
boundary尚清
shape规则
blood_flowFALSE
blood_flow_details未见明显血流信号
右侧乳腺信息
side右
duct_dilation[填写TRUE/FALSE，是否有导管扩张，若无则填写FALSE]
duct_dilation_details[填写导管扩张的具体描述，若无则留空]
blood_flow_present[填写TRUE/FALSE，是否存在异常血流信号]
blood_flow_details[填写血流信号的具体描述，如“未见明显血流信号”或具体血流描述]
nodules_present[填写TRUE/FALSE，是否存在结节]
nodules_count[填写结节数量，如3]
右侧乳腺结节 1
nodule_id3
location12点钟
distance_from_nipple_cmNULL
size_x_cm0.8
size_y_cm0.3
boundary清
shape规则
blood_flowFALSE
blood_flow_details未见明显血流信号
右侧乳腺结节 2
nodule_id4
location外上象限
distance_from_nipple_cmNULL
size_x_cm4.4
size_y_cm1.1
boundary尚清
shape欠规则
blood_flowFALSE
blood_flow_details未见明显血流信号
右侧乳腺结节 3
nodule_id5
location增生结节（疑）
distance_from_nipple_cmNULL
size_x_cm0.8
size_y_cm0.3
boundary清
shape规则
blood_flowFALSE
blood_flow_details#未见明显血流信号
结论
conclusion#左乳腺结节；右侧乳腺未见明显占位病变：双侧腋窝淋巴结。

示例输入：
{input}
输出模版：
{output_template}
输出示例：
```json
{output}
```

示例输入：
{input}

 输出要求：
    1、请将提取到字段填到输出模版的相应位置，请一定遵守，方便后期处理
    2、无法从示例输入中找到的信息就填NULL
    3、'本次输入'中没有的信息，不可以从'字段值信息解释'中提取
"""

## 胸部ct

In [70]:
ct_output = {
    "报告信息": {
        "report_id": "",
        "patient_id": "",
        "exam_date": "",
        "exam_location": "肺部",
        "肺部结构是否正常": "不正常"
    },
    "肺部结构信息": {
        "layer_structure": "",
        "airway_description": "",
        "mediastinal_findings": "",
        "cardiac_large_vessels": "",
        "pleural_and_cavity_findings": ""
    },
    "左侧肺部信息": {
        "side": "左",
        "nodules_present": "TRUE",
        "nodules_count": "2"
    },
    "左侧肺部结节 1": {
        "nodule_id": "1",
        "location": "左肺上叶尖后段胸膜下",
        "size_long_axis_cm": "0.49",
        "size_short_axis_cm": "0.54",
        "density": "磨玻璃",
        "shape": "",
        "suspected_diagnosis": ""
    },
    "左侧肺部结节 2": {
        "nodule_id": "2",
        "location": "左肺上叶尖后段",
        "size_long_axis_cm": "0.35",
        "size_short_axis_cm": "0.45",
        "density": "实性",
        "shape": "",
        "suspected_diagnosis": ""
    },
    "右侧肺部信息": {
        "side": "右",
        "nodules_present": "TRUE",
        "nodules_count": "1"
    },
    "右侧肺部结节 1": {
        "nodule_id": "3",
        "location": "右肺下叶外侧基底段胸膜下",
        "size_long_axis_cm": "0.16",
        "size_short_axis_cm": "0.27",
        "density": "实性",
        "shape": "",
        "suspected_diagnosis": ""
    },
    "结论": {
        "conclusion": "左肺上叶尖后段胸膜下见磨玻璃结节，左肺上叶尖后段及右肺下叶外侧基底段胸膜下见实性结节，边界清晰或尚可。"
    }
}

In [82]:
prompt_ct = """
任务说明：

您是一个医疗数据提取助手，专门用于解析肺部CT报告。您的任务是从给定的报告文本中提取以下结构化信息，并将其填写在指定的字典中。请确保信息的准确性和完整性，特别是在提取结节描述和诊断意见部分。

需要提取的信息类别：
1.报告信息：
•报告唯一标识（report_id）
•患者唯一标识（patient_id）
•检查日期（exam_date）
•检查部位（exam_location）
2.肺部结构正常性：
•肺部层次结构是否清晰，相关字段:layer_structure。
•气管及支气管的描述,相关字段:airway_description。
•纵隔结构是否正常,相关字段:mediastinal_findings。
•心脏及大血管的描述,相关字段:cardiac_large_vessels。
•胸膜及胸腔的描述,相关字段:pleural_and_cavity_findings。
3.肺部结节的结构化描述：
•结节所在侧别（左或右）。
•结节位置（如“左肺上叶”）,相应字段:location。
•结节大小（长轴、短轴，单位为厘米）,相应字段:size_long_axis_cm,size_short_axis_cm。
•结节形态描述（如“实性”）,相应字段:density。
•结节密度描述（如“微小/小”）,相应字段:shape。
•结节数量。
•结节的疑似诊断（如“肺大泡”）,相应字段:suspected_diagnosis。
4.肺部低密度区的描述：
•低密度区所在侧别。
•低密度区位置,相应字段:location。
•低密度区大小（长轴、宽轴、高轴，单位为厘米）相应字段:size_long_axis_cm,size_long_axis_cm,size_height_axis_cm。
•低密度区形态描述（如“囊状透光影”）,相应字段:shape。
•低密度区的疑似诊断（如“肺大泡”）,相应字段:suspected_diagnosis。
5.报告结论的结构化描述(相关字段:conclusion)：
•总结报告中的结论部分,示例('报告结论':'左肺多发微、小结节, 建议复查;右肺下叶后基底段肺大泡','结构化描述':'左肺多发微、小结节；右肺下叶后基底段肺大泡')

表格格式要求：

将所有提取的信息按照以下垂直排列的表格格式填写。每个字段在表格的第一列，相关值在第二列,字段和相关值用#连接。对于存在多个结节或低密度区的情况，使用编号区分每个结节或低密度区的信息。

字段值信息解释：
报告信息
report_id[填写报告唯一标识，例如1]
patient_id[填写患者唯一标识，例如1001]
exam_date[填写检查日期，例如2024-12-17]
exam_location[填写检查部位，例如“肺部”]
肺部结构是否正常[填写“正常”或“不正常”，根据肺部结构描述]
肺部结构信息
layer_structure[填写肺部层次结构描述，例如“纵隔结构清晰”或“纵隔结构模糊”]
airway_description[填写气管及支气管的描述，例如“气管正常”或“气管小憩室”]
mediastinal_findings[填写纵隔结构的描述，例如“未见增大淋巴结”]
cardiac_large_vessels[填写心脏及大血管的描述，例如“心脏及大血管未见异常”]
pleural_and_cavity_findings[填写胸膜及胸腔的描述，例如“左侧胸膜局部增厚，双侧胸腔未见积液”]
左侧肺部信息
side左
nodules_present[填写TRUE/FALSE，是否存在结节]
nodules_count[填写结节数量，如1]
左侧肺部结节 1
nodule_id1
location左肺上叶
size_long_axis_cm0.3-0.6
size_short_axis_cm[如适用，可填写]
density实性
shape线状
suspected_diagnosis[如适用，可填写，如“肺大泡”]
左侧肺部低密度区 1
low_density_id1
location左肺上叶胸膜下
size_long_axis_cm4.4
size_width_axis_cm1.1
size_height_axis_cm3.5
shape囊状透光影
suspected_diagnosis[如适用，可填写，如“肺大泡”]
右侧肺部信息
side右
nodules_present[填写TRUE/FALSE，是否存在结节]
nodules_count[填写结节数量，如3]
右侧肺部结节 1
nodule_id2
location右肺上叶前段
size_long_axis_cm0.3-0.6
size_short_axis_cmNULL
density实性
shape微小/小
suspected_diagnosisNULL
右侧肺部结节 2
nodule_id3
location左肺上叶舌段
size_long_axis_cm0.3-0.6
size_short_axis_cmNULL
density实性
shape微小/小
suspected_diagnosisNULL
右侧肺部结节 3
nodule_id#4
location#右肺下叶外基底段(im116/147)
size_long_axis_cmNULL
size_short_axis_cmNULL
density#实性
shape#微小/小
suspected_diagnosis#NULL
结论
conclusion#两肺纹理稍增重、紊乱；肺上叶陈旧性病；主动脉及冠脉走行区部分管壁钙；肝囊肿


输出模版：
{output_template}
输出示例：
```json
{output}
```

本次输入：
{input}

输出要求：
    1、请将提取到字段填到输出模版的相应位置，请一定遵守，方便后期处理
    2、无法从示例输入中找到的信息就填NULL
    3、'本次输入'中没有的信息，不可以从'字段值信息解释'中提取
"""

## 检查报告content

In [77]:
input_text_rx_list = ['双侧乳腺腺体内部结构稍紊乱，回声欠均匀,CDFI:腺体内未见异常血流信号+双侧乳腺增生,符合BI-RADS[2]类',
            '双侧乳腺腺体结构轻度紊乱、回声欠均匀。右侧乳腺约9点钟位置腺体内可见3.6x1.5mm囊性回声，形态规整。左侧乳腺约2-3点钟位置乳头旁腺体内可见9.3x4.0mm低回声区+右侧乳腺囊性结节（BI--RADS2类）；左侧乳腺低回声区（考虑腺病、建议复查）',
            '双侧乳腺腺体组织强弱相间，回声不均乳，导管不扩张，左乳腺9点方向见一7.5mm*4.3mm稍低回声结节，边界清，CDFI未见明显血流信号，双侧腋窝探及，淋巴结回声，左侧约9.1mm*4.9mm，右侧约7.8mm*3.4mm，CDFI，未探及异常血流+左乳腺结节BI-RADS3类；右侧乳腺未见明显占位病变；双侧腋窝淋巴结',
            '双乳腺层次结构清晰,腺体排列尚可,左侧乳晕下导管无扩张。左侧乳腺可见多个低回声结节节，较大者位于11约点钟位置距乳头约2.5cm处腺体层内，大小约0.4×0.2cm，界清，形态规则，周边及内部未见明显血流信号。右侧乳腺可见数支导管局限性增宽，较宽处内径约0.23cm，内透声可。双侧腋下探查：未见明显异常肿大淋巴结回声。双侧锁骨上探查：未见明显异常占位回声。CDFI：未见异常血流信号+左侧乳腺多发低回声结节--BIRADSIII类;右侧乳腺局部导管扩张',
            '右乳见散在囊性回声结节，大者位于9点距乳头约2.1cm处，大小约0.4cmx0.3cmx0.2cm界清，形态规则，内透声尚可，内未探及血流信号；余双侧乳腺层次清晰,腺体不厚,结构紊乱，腺导管未见扩张，内未见明确异常血流信号。双侧腋下未见明显肿大淋巴结。+右乳囊性回声结节。BI-RADS分级2级;双乳结构不良声像图改变。',
            '双侧乳腺腺体结构轻度紊乱、回声欠均匀。左侧乳腺约9-10点钟位置腺体内可见10.7x3.5mm囊性回声，形态规整。右侧乳腺约12-1点钟位置腺体内可见29.4x21.4mm低回声，形态不规整,内部回声不均匀。CDFI:其内可见条状血流信号+左侧乳腺囊性结节（BI--RADS2类)。右侧乳腺低回声包块（BI--RADS4a类',
            '双侧乳房切面形态，轮廓正常体积增大，边界光滑完整，内部回声增强，乳腺内部结构紊乱，分布不均，呈粗大的光点光斑及高低不等的海绵状回声，于右侧乳腺相当于时钟10点处，可探及一低回声结节大小约0.6cm×0.3cm，边界光滑、清晰，形态规则，内部回声均匀，后方回声无衰减，CDFI显示双乳未见明显异常血流信号+双侧乳腺增生；右侧乳腺结节',
            '双侧乳腺腺体形态正常，结构清晰,双乳可见多个低回声团块，左乳大的约1.5×0.7cm，右乳大的约1.6×0.9cm，境界清,有包膜，形态规则，内回声均质，CDFI检及点条状血流信号.双乳腋下引流区探查未见明显肿大淋巴结声像+双乳多发实质性肿块（BI-RADS3）',
            '双乳腺腺体层稍厚，腺体结构紊乱，组织分布不均匀，未见乳导管局限性扩张，右乳2点方向可见大小约4.5×4.3×4.0mm低回声结节，边界欠清，表面可见钙化，CDFI未见明显血流信号双，腋下及锁骨下，未探及明显异常肿大淋巴结+右乳结节（BI-RADS3类）',
            '双侧乳腺隆乳术后改变，隆乳区边界尚光整，内透声好。乳腺腺体不厚，结构正常，回声尚均匀，未见明显囊、实性占位性病变。CDFI:双侧乳腺未见明显异常血流信号。左侧腋窝可探及稍低回声区，大小约10.6mmx9.3mm，椭圆形，轮廓清晰，边界光滑，内部回声稍增强。CDFI:未见明显异常血流信号。右侧腋窝未见明显肿大淋巴结。双侧锁骨下未探及明显肿大淋巴结。+双乳腺隆乳术后,未见明显占位声像,左侧腋窝淋巴结可见,请定期复查',
            '双乳皮肤和皮下脂肪层回声清楚，未见异常，双乳腺体层结构紊乱，回声呈强弱相间，分布不均匀，导管无扩张。右乳约9点钟方向距乳头3.3cm腺体层内低回声结节0.3x0.2cm，轮廓清晰，边界规整。左乳约8点钟方向距乳头3.2cm腺体层内低回声结节0.3x0.2cm，轮廓清晰，边界规整，5点钟方向距乳头1.1cm腺体层内低回声结节0.4x0.4x0.3cm，11点半方向距乳头4.4cm腺体层内低回声结节0.3x0.2cm，局部似与脂肪组织相连，形态不规则，略呈纵向生长，边界尚清。CDFI双乳腺无异常血流信号。双侧腋下未见明显异常肿大淋巴结+双乳低回声结节BI-RADS3类；左乳低回声结节BI-RADS4a类',
            '双侧乳房切面形态轮廓正常，层次清楚，边界光滑完整，内部回声增强，结构紊乱，分布不均，局部腺体增厚，回声减低，右侧乳腺12点方向可见一个大小约7. 1X4.6mm的无回声结节，边界尚清，左侧乳腺可见几个大小不等，最大约8. 7X6mm (12点方向)的低回声结节，边界尚清。双侧腋窝扫查未见明显异常肿大淋巴结回声。+右侧乳腺内囊性结节,符合BI-RADS2类,左侧乳腺内低回声结节,符合BI-RADS3类',
            '双侧乳腺组织厚度正常，内部回声均匀，乳腺导管未见扩张，双侧乳腺未见确切占位，双侧腋窝未见确切异常淋巴结+未见明显异常',
            "双侧乳腺组织层次清楚,乳腺组织回声增强,光点增粗,排列紊乱,内见条索状低回声带,呈豹纹征双侧乳腺见多个低回声,边缘规则,内部回声均匀,左侧乳腺较大的7x4mm位于2点;右侧乳腺较大的8x6nm.位于10点。右侧乳腺8点处可见12x8mm无回声边界清,透声好。双侧腋下未见明显肿大淋巴结。CDFI双侧乳腺低回声周边及内、右侧乳腺无回声未见明显血流信号显示。+1、双侧乳腺低回声,BI-RADS3类建议定期复查。2、右侧乳腺无回声,BI-RADS2类。",
            '两侧乳房形态饱满,层次清楚,腺体稍增厚,腺体内回声强弱相间,分布欠均匀。双侧乳腺导管扩张,最大内径分别约2.8mm(右侧)、3.2mm(左侧),管壁光滑,管腔内透声好,未见明显异常团状回声。右乳腺9点钟见一囊性暗区,大小约7×6mm,边清,透声尚可。CDFI显示暗区血供0级;双侧乳腺内见较丰富血流信号。双侧腋窝探查未见明显异常回声+哺乳期乳腺声像，右乳腺囊肿，乳腺BI-RADS分级(2类)',
            '双侧乳腺对称,外形正常。右侧乳腺见1个类椭圆形低回声肿块,大小约5.7mm×3.3mm(10点钟左右方向距离乳头约35mm)边界清,稍欠规整,纵横比＜1,内回声分布尚均匀,未见点状强回声,后方回声增强。CDFII:肿块周边及内部未见明显彩流信号。左侧乳腺见数个类椭圆形囊性暗区其中一个大小约2.9mm×2.4mm(8点钟左右方向),囊壁薄,光滑,规整,可见侧壁声影,暗区内透声尚可,后方回声增强。CDFI暗区周边及内部未见明显彩流信号显示乳腺其余组织结构欠清晰,腺组织欠均质,呈细蜂窝状低回声。乳腺导管未见增宽+右侧乳腺实性局灶性病变(性质待定)未排增生结节或其它?(BI-RADS分类，4A类)，左侧乳腺囊性局灶性病变(BI-RADS分类2类)，建议6个月左右复查',
            '双侧乳房切面形态轮廓正常,腺体层回声增强增粗,结构稍紊乱,双侧乳腺导管未见扩张。左侧乳腺腺体内探及一低回声结节,大小约5.8mmx2.7mm(1点距乳头1cm)边界尚清,尚规则,内回声欠均匀。CDFI低回声结节周边及内部未见明显异常血流信号CDFI乳腺腺体内未见明显异常彩色血流+双侧乳腺小叶增生，左侧乳腺低回声结节(增生结节)BI-RADS3级，BI-RADS0级影像学评估不完全,需要进一步评估(临床有体征,超声检查无征象者)1，级阴性发现(常规体检一年一次)2级良性发现(6个月到1年复查一次)3级可能良性发，现(恶性可能＜2%(建议定期内随访3-6个月复查)4级4a级低度可疑恶性(恶性可能，3%-8%)(3个月复查);4h级中度可疑恶性(恶性可能9%-49%(活检)4c级高度可疑恶性，(恶性可能50%-95%)(手术)5级典型恶性征象(恶性可能≥95%(几乎认定ca手术处，理)6级已行活检证实的恶性肿瘤',
            '双侧乳腺结构层次清晰,乳腺导管增粗乳腺组织回声紊乱,见增粗的片索状回声减低区与稍高回声相间,未见肿块回声。右乳10点钟方向导管局限性扩张,前后径0.2cm双侧腋窝未见明显异常淋巴结回声CDFI未见明显异常血流信号。+双侧乳腺小叶增生(BI-RADS1类)，右乳导管局限性扩张',
            '右侧乳腺切除术后;右侧胸壁皮下软组织未见明显异常的实性及囊性包块CDFI未见明显异常血流信 号。 左侧乳腺各组织结构清晰,腺组织较均质,呈细蜂窝状较强回声+右侧乳腺切除术后',
            '双乳腺体层显示清晰，回声呈强弱相间,分布欠均匀,呈粗大点状及斑片状，左乳部分导管扩张,内径2.6mm,右乳见数个大小不等的无回声，其中最大一个位于7点钟方向乳头旁，大小约8.9mm×6.1mm,彩色多普勒检查未见明显血流信号：右乳11点钟方向距乳头5mm处见大小约5.5mm×3.0mm低回声，边界清楚,形态规则,彩色多普勒检查未见明显血流信号。左乳见数个低回声,其中一个位于11点钟方向距乳头30mm处，大小约5mm×3.3mm,边界滴楚,形态规则,彩色多普勒检查未见明显血流信号显示+双乳异常低回声,BI-RADS3类：。&amp;&amp;双乳增生图像,右乳多发性囊肿,左乳部分导管扩张。',
            '双侧乳腺轻度增生。左侧乳腺低回声结节（大小约0.5cm×0.3cm，位于10点钟方向）+双侧乳腺轻度增生；左侧乳腺低回声结节',
            '双侧乳房切面形态轮廓正常，腺体层结构紊乱回声不均呈条素状改变。右侧乳腺局部导管扩张。内径最觉处约2.9mm右乳12点方向探及大小约8x5mm囊性结节，包膜完整,运声好,其内可见分隔,CDFI:未见异常血流信号+2.双侧乳腺增生。3.右侧乳腺导管扩张;4.右侧乳腺囊肿。',
            '左侧乳腺囊性结节（大小约0.4cm×0.2cm，位于2点钟方向,边界清晰,形态规则) 双侧乳腺轻度增生。 *科室小结左侧乳腺囊性结节。 双侧乳腺轻度增生+左侧乳腺囊性结节&&双侧乳腺轻度增生',
            '左侧乳腺低回声结节（大小约0.9cm×0.6cm，位于2点钟方向,边界清晰,形态规则,内回声均匀;右侧乳腺低回声结节（大小约0.4cm×0.3cm,位于9点钟方向,边界清晰,形态规则,内回声均匀+双侧乳腺轻度增生;双侧乳腺低回声结节。',
            '双侧乳腺层次清楚：双侧乳腺腺体不厚腺体回声杂乱：分布不均：双侧乳腺均见数个类似脂防样低回声区：右侧乳腺外下象限脂肪层见大小约5mmXm稍高回声区CDFI周边及其内未见明显血董+双侧乳腺腺体回声不均匀;右侧乳腺稍高回声区',
            '双侧乳腺组织结构清晰，双侧乳腺导管未见扩张。右乳内上象限可见大小约0.64*0.55cm低回声结节,边界清,形态规则。CDFI:双侧乳腺未见明显异常彩色血流信号+右侧乳腺低回声结节(BI-RADS4类a)',
            '双侧乳房对称,结构层次清楚乳腺组织无增厚内回声增强增粗不均匀乳腺管无明显扩张右侧乳腺12点位A区深层见-0.4×0.3cm低回声结节，界清。CDFI显示双侧乳腺组织未见明显异常血流信号+双侧乳腺增生右侧乳腺低回声结节（建议定期门诊复查）、双侧乳腺增生',
            '双侧乳腺组织萎缩变薄，部分乳腺组织回声紊乱左侧乳腺外上象限可见边界清晰的低回声区，大小约4×3mm。双侧腋下未见明显肿大淋巴结。CDFI:血流未见明显异常+双乳腺老年性退化改变伴局部增生声像图；左侧乳腺结节待查',
            '双乳腺扫查显示：双乳腺腺体不均匀性增厚，结构紊乱,条索状偏强回声与偏低回声 相兼。于双乳均可见多个囊性回声大者位于左侧，大小约1.0*0.8cm,内透声可。CDFI: (-)+双侧乳腺增生并双乳多发囊性回声（定期复查）',
            '双侧乳腺扫查：。 双侧乳腺内部回声不均，呈豹纹征改变,右侧乳腺内约9点钟处见大小约0.74X0.29cm无回声 区,边界清,左侧乳腺内未见明显局限性异常回声。 双乳内未见异常血流信号+双侧乳腺呈增生样改变&&右侧乳腺囊性结节',
            '腺体回声减低、欠均质，腺体层内见较多管状无回声，以乳头为中心呈放射状分布，较宽处约3.4mm，CDFI内未见明显彩色血流信号。+双乳腺体层内未见明显占位性病变（BI-RADS1类）',
            '乳腺超声：双侧乳腺腺体层轻度增厚，内部结构稍紊乱,回声欠均匀,CDFI:腺体内未见异常血流信号。注：受内分泌及仪器等因素影响，结节的存在与否,大小,位置,血供分类情况均可能发生变化，部分乳管内小占位超声检查不到，如有不适,请进一步检查。+双侧乳腺轻度增生',
            '乳腺超声：绝经期,双侧乳腺腺体层增厚，内部结构紊乱,回声不均,右乳头上方局部腺体回声减低。左侧乳腺腺体层内见低回声，大小约3mm×2mm（12点位），边界清,形态规则,未见血流。CDFI：内未见异常血流信号+双侧乳腺退化不全;左侧乳腺结节（BI-RADS3类）',
            '双侧乳腺层次较清晰,结构排列较整齐,未见囊、实性占位病变。CDFI:双侧乳腺内未见异常血 流信号+双侧乳腺未见明显肿块,拟BI-RADS-US:1类。']

input_text_ct_list = ["两侧胸廓对称。肺窗示两肺纹理稍增重、紊乱，右肺及左肺上叶尖段可见条索样、小结节样及斑片状高密度影，部分边界清晰，大部分病灶内可见钙化影。两侧肺门不大。纵隔窗示心影及大血管形态正常，主动脉壁及双侧冠状动脉走形区可见斑块状钙化影，纵隔内未见肿块及明显肿大淋巴结。无胸腔积液及胸膜增厚；扫描层面肝脏区域内可见小类圆形低密度影，边界尚清晰。小结意见：1、两肺纹理稍增重、紊乱，请结合临床。2、考虑右肺及左肺上叶陈旧性病变，请结合临床。3、主动脉及冠脉走行区部分管壁钙化。 4、附见肝囊肿。",
    "胸廓对称, 左侧第6、7 肋骨前支骨质形态不规则并骨痂生长双肺实质未见明显异常, 双肺纹理未见异常, 双肺门结构正常。纵隔居中, 纵隔未见肿大淋巴结, 气管、支气管、心脏显示正常胸腔无积气、积液征象, 胸膜无增厚。扫描区肝、脾未见异常 + 双肺CT平扫未见明显异常。左侧第6、7肋骨前支陈旧性骨折",
    "两肺纹理稍增多，双肺下叶示微小磨玻璃结节影，较大者位于右肺下旁、直径约3.9mm;右肺见多发粟粒灶，边界尚清;右肺示条索状密度增高欠清，纵隔居中，纵隔内示多发中小淋巴结影，边界尚清，主气管及双侧支畅，心影血管界面清晰，双侧胸腔未见明显积液。影像学诊断: 1、双肺下叶磨玻璃结节灯，右肺粟粒灶，右肺纤维增殖灶，随诊。2、纵隔多发中小淋巴结，随诊。",
    "[肺CT]所见，胸廓对称, 双肺纹理清晰, 走形自然, 肺野透光度良好, 右胸壁见小高密度，结节样病灶, 直径约3 * 5mm边缘较清。右肺见一磨玻璃密度，结节影, 直径约8x7mm, 边缘欠规则。双胸壁见少许纤维条索影。余肺未见明显异常征象。双肺门不大, 纵膈无偏移, 心脏及大血管显示形态正常, 纵膈内未见肿块及肿大淋巴结 + 右肺磨玻璃密度结节建议去医院进一步检查，双侧陈旧性胸膜粘连",
    "胸廓对称, 气管纵隔居中, 两肺纹理增强, 透光度增强, 双肺内少许微小斑点结节影，ima19层右肺中叶外侧小于0.2cm结节、肺门结构紊乱，纵隔内结构显示清晰，胸膜无增厚 + 右肺小结节影, 请结合临床病史, 建议定期复查、必要时进一步检查。",
    "两侧胸廓对称左肺上叶尖后段胸膜下（Se3Im28）见一大小约4.9X5.4mm的磨玻璃结节影, 边界尚可左肺上叶尖后段（Se3Im83）见一大小约3.5X4.5mm的结节影，边界尚可右肺下叶外侧基底段胸膜下（Se3Im172）见一大小约1.6X2.7mm的结节影，边界清晰;",
    "左肺下叶后侧基底段胸膜下（Se3Im175）见一大小约2.8X3.6mm的结节影，边界清晰左肺上叶下舌段见条索状密度增高影，边界清晰余肺纹理清晰、规则，未见明显实质性病变影所见气管及支气管通畅纵隔内及两侧腋窝下未见明显肿大淋巴结影；两侧胸腔内未见明显积液征象；心影形态大小尚可 + 两肺散在结节影, 与2022 - 4 - 16CT大致相仿，请随访。 & amp; & amp;左肺上叶下舌段纤维灶。",
    "双侧胸廓对称，两肺支气管血管束走行规则，双肺上叶见斑片状及条索状影高密度影，边界清楚，纵膈窗示其内可见斑片状钙化影，肺门形态大小及位置未见异常气管支气管通畅纵膈窗示纵膈内未见增大淋巴结影心影大小形态未见明显异常双侧胸膜增厚局部可见钙化影胸膜腔内未见积液三维重建更加立体直观清晰显示胸部情况。所见肝实质密度减低CT值约40HU低于同层面脾脏密度CT值约48HU + 双肺上叶陈旧性病变；双侧胸膜增厚、局部钙化；轻度脂肪肝",
    "两侧胸廓对称。肺窗示左肺上叶见微小结节灶，直径约3mm边界清右下肺少许索条影余两肺野纹理清晰, 未见明显异常密度影。两侧肺门不大。纵隔窗示心影及大血管形态正常, 纵隔内未见肿块及明显肿大淋巴结。无胸腔积液及胸膜增厚。+肺结节影，肺纤维灶",
    "双侧胸廓对称, 左肺上叶（IM40、IM107）见多发实性结节影，长径范围约3 - 5mm, 较大者大小约5mm×5mm，位于左肺上叶上舌段。右肺下叶后基底段可见囊性低密度影, 边界清晰：气管及双侧支气管开口通畅，双侧肺门不大, 纵隔居中, 纵隔及双侧肺门未见肿大淋巴结。心脏不大。双侧胸腔未见液体密度影 + 左肺多发微、小结节, 建议复查 & amp; & amp;右肺下叶后基底段肺大泡"]

print("ct case num:",len(input_text_ct_list))
print("rx case num:",len(input_text_rx_list))

ct case num: 10
rx case num: 34


## 腾讯混元

In [47]:
import os
from openai import OpenAI

def qa_hunyuan(prompt_input_ct):

    # 构造 client
    client = OpenAI(
        api_key="sk-lhv98a6hmq3frLhgY1MvSlTcKnp38cBEnvAgAjzjKrAyiOym", # 混元 APIKey
        base_url="https://api.hunyuan.cloud.tencent.com/v1", # 混元 endpoint
    )


    # 自定义参数传参示例
    completion = client.chat.completions.create(
        model="hunyuan-pro",
        messages=[
            {
                "role": "user",
                "content": prompt_input_ct
            },
        ],
        extra_body={
            "enable_enhancement": True, # <- 自定义参数
        },
    )
    return completion.choices[0].message.content

## 百川API

In [9]:
import requests
import json


def qa_baichuan(input_):
    # 定义API的URL
    url = "https://api.baichuan-ai.com/v1/chat/completions"
    baichuan_key = "sk-4ce3bfe548d2bd5209aee1e19f72f4b8"

    # 定义请求头
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {baichuan_key}"  # 请确保你已经定义了API_KEY变量
    }

    # 定义请求体
    data = {
        "model": "Baichuan4",
        "messages":[
            {"role": "user",
        "content": input_}
        ],
        "temperature": 0.3,
        "stream": False
    }

    # 发送POST请求
    # response = requests.post(url, headers=headers, data=json.dumps(data))
    response = requests.post(url, headers=headers, json=data)


    # 检查响应状态码
    if response.status_code == 200:
        # 如果请求成功，打印响应内容
        # print(response.json())
        return response.json()["choices"][0]["message"]["content"]
    else:
        # 如果请求失败，打印错误信息
        print(f"Request failed with status code {response.status_code}")

## 进行测试

In [14]:
# import concurrent.futures

# def run_qa(input_text):
#     prompt_input = prompt_rx.format(input=input_text)
#     return qa_hunyuan(prompt_input), qa_baichuan(prompt_input)

# # 假设 input_text_rx_list 是你要处理的输入列表
# input_text_rx_list = [...]  # 你的输入列表

# # 使用 ThreadPoolExecutor 来并行运行函数
# with concurrent.futures.ThreadPoolExecutor() as executor:
#     futures = [executor.submit(run_qa, input_) for input_ in input_text_rx_list]
    
#     for idx, future in enumerate(concurrent.futures.as_completed(futures)):
#         answer_hunyuan, answer_bc = future.result()
#         print(f"case num: {idx}")
#         result_dict = {"answer_hunyuan": answer_hunyuan, "answer_bc": answer_bc}
#         print("llm混元 答案生成...", answer_hunyuan)
#         print("llm百川 答案生成...", answer_bc)
#         # 你可以在这里处理 result_dict 或者做其他事情

In [83]:
#乳腺报告
print("乳腺报告结构提取。。。")
result_rx = []
for idx,input_ in enumerate(input_text_rx_list[:3]):
    print(f"case num:{idx}")
    result_dict = {}
    prompt_input_ = prompt_rx.format(input=input_,output_template=output_struct_rx_dict,output=rx_output)
    # print(prompt_input_)
    print("llm混元 答案生成...")
    answer_hunyuan = qa_hunyuan(prompt_input_)
    # print("llm百川 答案生成...")
    # answer_bc = qa_baichuan(prompt_input_)
    result_dict["description"] = input_
    # result_dict["ans_baichuan"] = answer_bc
    result_dict["ans_hunyuan"] = answer_hunyuan
    result_rx.append(result_dict)
    

乳腺报告结构提取。。。
case num:0
llm混元 答案生成...
case num:1
llm混元 答案生成...
case num:2
llm混元 答案生成...


In [48]:
#胸部ct报告
print("胸部ct报告结构提取。。。")
result_ct = []
for idx, input_ in enumerate(input_text_ct_list):
    print(f"case num:{idx}")
    result_dict = {}
    prompt_input_ct = prompt_ct.format(input=input_,output_template=output_struct_ct_dict)
    print("llm混元 答案生成...")
    answer_hunyuan = qa_hunyuan(prompt_input_ct)
    # print("llm百川 答案生成...")
    # answer_bc = qa_baichuan(prompt_input_ct)
    result_dict["description"] = input_
    # result_dict["ans_baichuan"] = answer_bc
    result_dict["ans_hunyuan"] = answer_hunyuan
    result_ct.append(result_dict)

胸部ct报告结构提取。。。
case num:0
llm混元 答案生成...
case num:1
llm混元 答案生成...
case num:2
llm混元 答案生成...
case num:3
llm混元 答案生成...
case num:4
llm混元 答案生成...
case num:5
llm混元 答案生成...
case num:6
llm混元 答案生成...
case num:7
llm混元 答案生成...
case num:8
llm混元 答案生成...
case num:9
llm混元 答案生成...


In [55]:
for idx, one_ct in enumerate(result_ct):
    print("index:",idx)
    print(one_ct["ans_hunyuan"])

index: 0
```json
{
    "报告信息": {
        "report_id": "1",
        "patient_id": "1001",
        "exam_date": "2024-12-17",
        "exam_location": "肺部",
        "肺部结构是否正常": "不正常"
    },
    "肺部结构信息": {
        "layer_structure": "两肺纹理稍增重、紊乱",
        "airway_description": "两侧肺门不大",
        "mediastinal_findings": "纵隔内未见肿块及明显肿大淋巴结",
        "cardiac_large_vessels": "心影及大血管形态正常，主动脉壁及双侧冠状动脉走形区可见斑块状钙化影",
        "pleural_and_cavity_findings": "无胸腔积液及胸膜增厚"
    },
    "左侧肺部信息": {
        "side": "左",
        "nodules_present": "TRUE",
        "nodules_count": "1"
    },
    "左侧肺部结节 1": {
        "nodule_id": "1",
        "location": "左肺上叶尖段",
        "size_long_axis_cm": "0.3-0.6",
        "size_short_axis_cm": "NULL",
        "density": "实性",
        "shape": "小结节样",
        "suspected_diagnosis": "陈旧性病变"
    },
    "右侧肺部信息": {
        "side": "右",
        "nodules_present": "TRUE",
        "nodules_count": "1"
    },
    "右侧肺部结节 1": {
        "nodule_id": "2",
        "location": "右肺上叶尖段

In [49]:
# #output结果存储
result_all = {
    # "ruxian":result_rx,
    "ct":result_ct
}
with open("results/outputs/results_rx_ct_v3.json","w")as f:
    json.dump(result_all,f,ensure_ascii=False,indent=2)

In [23]:
len(result_ct)

10

## 答案整理

In [22]:
def past_out(ans_bc_values,output_struct_ct):
    dict_key = {}
    for k,v in output_struct_ct.items():
        v_dict = {}
        for k_y in v:
            v_dict[k_y] = "NULL"
        dict_key[k] = v_dict
    if "报告信息" not in ans_bc_values:
        big_key = "报告信息"
    else:
        big_key = ""
    for v in ans_bc_values:
        print(v)
        if "#" not in v:
            big_key = v
        else:
            # print(big_key)
            split_vs  = v.split("#")
            if len(split_vs) !=2:
                continue
            k_,v_ = split_vs[0], split_vs[1]
            if k_ in dict_key[big_key].keys():
                dict_key[big_key][k_]=v_
    #clear dict_key
    clear_dict_data = {}
    for k,v in dict_key.items():
        # print(k)
        # print(v)
        values = list((set(v.values())))
        if len(values)==1 and values[0]=="NULL":
            continue
        clear_dict_data[k] = v
    
    # return clear_dict_data
    return dict_key


In [35]:
for res_ct in result_ct:
    print(res_ct)

{'description': '两侧胸廓对称。肺窗示两肺纹理稍增重、紊乱，右肺及左肺上叶尖段可见条索样、小结节样及斑片状高密度影，部分边界清晰，大部分病灶内可见钙化影。两侧肺门不大。纵隔窗示心影及大血管形态正常，主动脉壁及双侧冠状动脉走形区可见斑块状钙化影，纵隔内未见肿块及明显肿大淋巴结。无胸腔积液及胸膜增厚；扫描层面肝脏区域内可见小类圆形低密度影，边界尚清晰。小结意见：1、两肺纹理稍增重、紊乱，请结合临床。2、考虑右肺及左肺上叶陈旧性病变，请结合临床。3、主动脉及冠脉走行区部分管壁钙化。 4、附见肝囊肿。', 'ans_hunyuan': '|字段|值|\n|--|--|\n|report_id#1|\n|patient_id#NULL|\n|exam_date#NULL|\n|exam_location#肺部|\n|layer_structure#两肺纹理稍增重、紊乱|\n|airway_description#两侧肺门不大|\n|mediastinal_findings#纵隔内未见肿块及明显肿大淋巴结|\n|cardiac_large_vessels#心影及大血管形态正常，主动脉壁及双侧冠状动脉走形区可见斑块状钙化影|\n|pleural_and_cavity_findings#无胸腔积液及胸膜增厚|\n|side左| |\n|nodules_present#TRUE|\n|nodules_count#1|\n|左侧肺部结节 1# |\n|nodule_id1#1|\n|location#左肺上叶尖段|\n|size_long_axis_cm#NULL|\n|size_short_axis_cm#NULL|\n|density#实性|\n|shape#小结节样|\n|suspected_diagnosis#陈旧性病变|\n|左侧肺部低密度区 1#NULL|\n|low_density_id1#NULL|\n|location#NULL|\n|size_long_axis_cm#NULL|\n|size_width_axis_cm#NULL|\n|size_height_axis_cm#NULL|\n|shape#NULL|\n|suspected_diagnosis#NULL|\n|side右| |\n|nodules_present#TRUE|\n|nod

In [34]:
result_ct[2]

{'description': '两肺纹理稍增多，双肺下叶示微小磨玻璃结节影，较大者位于右肺下旁、直径约3.9mm;右肺见多发粟粒灶，边界尚清;右肺示条索状密度增高欠清，纵隔居中，纵隔内示多发中小淋巴结影，边界尚清，主气管及双侧支畅，心影血管界面清晰，双侧胸腔未见明显积液。影像学诊断: 1、双肺下叶磨玻璃结节灯，右肺粟粒灶，右肺纤维增殖灶，随诊。2、纵隔多发中小淋巴结，随诊。',
 'ans_hunyuan': '|字段|值|\n|----|----|\n|report_id|1|\n|patient_id|NULL|\n|exam_date|NULL|\n|exam_location|肺部|\n|layer_structure|两肺纹理稍增多|\n|airway_description|主气管及双侧支畅|\n|mediastinal_findings|纵隔居中，纵隔内示多发中小淋巴结影，边界尚清|\n|cardiac_large_vessels|心影血管界面清晰|\n|pleural_and_cavity_findings|双侧胸腔未见明显积液|\n|左侧肺部信息#side左|TRUE|\n|nodules_present#TRUE|\n|nodules_count#1|\n|左侧肺部结节 1#nodule_id1|\n|location#左肺下叶|\n|size_long_axis_cm#约3.9mm|\n|size_short_axis_cm#NULL|\n|density#磨玻璃|\n|shape#微小|\n|suspected_diagnosis#磨玻璃结节|\n|右侧肺部信息#side右|TRUE|\n|nodules_present#TRUE|\n|nodules_count#多发|\n|右侧肺部结节 1#nodule_id2|\n|location#右肺下旁|\n|size_long_axis_cm#约3.9mm|\n|size_short_axis_cm#NULL|\n|density#磨玻璃|\n|shape#微小|\n|suspected_diagnosis#磨玻璃结节|\n|右侧肺部结节 2#nodule_id3|\n|location#右肺（粟粒灶位置未明确）|\n|size_long_axis_cm#NULL|\n|size_short_

In [32]:
# ct答案整理
ct_csv = {}
for idx, rsult_ in enumerate(result_ct):
    print(rsult_)
    ct_csv[idx]={}
    ct_csv[idx]["description"] = rsult_["description"]
    # ans_bc = rsult_["ans_baichuan"]
    # output_struct_ct_cp = output_struct_ct.copy()
    # keyword = list(output_struct_ct_cp.keys())
    # print(ans_bc.split("\n"))
    # ans_bc_values = [v.strip("：") for v in ans_bc.split("\n") if "#"in v or v.strip("：") in keyword]
    # print(ans_bc_values)
    # print("baichuan prope")
    # ct_csv[idx]["baichuan"] = past_out(ans_bc_values,output_struct_ct_cp)

    ans_bc = rsult_["ans_hunyuan"]
    output_struct_ct_cp = output_struct_ct.copy()
    keyword = list(output_struct_ct_cp.keys())
    print(("报告信息\n"+ans_bc.strip("```")))
    ans_bc_values = [v for v in ("报告信息\n"+ans_bc.strip("```").strip("|")).split("\n") if "#"in v or v in keyword]
    ct_csv[idx]["hunyuan"] = past_out(ans_bc_values,output_struct_ct_cp)


{'description': '两侧胸廓对称。肺窗示两肺纹理稍增重、紊乱，右肺及左肺上叶尖段可见条索样、小结节样及斑片状高密度影，部分边界清晰，大部分病灶内可见钙化影。两侧肺门不大。纵隔窗示心影及大血管形态正常，主动脉壁及双侧冠状动脉走形区可见斑块状钙化影，纵隔内未见肿块及明显肿大淋巴结。无胸腔积液及胸膜增厚；扫描层面肝脏区域内可见小类圆形低密度影，边界尚清晰。小结意见：1、两肺纹理稍增重、紊乱，请结合临床。2、考虑右肺及左肺上叶陈旧性病变，请结合临床。3、主动脉及冠脉走行区部分管壁钙化。 4、附见肝囊肿。', 'ans_hunyuan': '|字段|值|\n|--|--|\n|report_id#1|\n|patient_id#NULL|\n|exam_date#NULL|\n|exam_location#肺部|\n|layer_structure#两肺纹理稍增重、紊乱|\n|airway_description#两侧肺门不大|\n|mediastinal_findings#纵隔内未见肿块及明显肿大淋巴结|\n|cardiac_large_vessels#心影及大血管形态正常，主动脉壁及双侧冠状动脉走形区可见斑块状钙化影|\n|pleural_and_cavity_findings#无胸腔积液及胸膜增厚|\n|side左| |\n|nodules_present#TRUE|\n|nodules_count#1|\n|左侧肺部结节 1# |\n|nodule_id1#1|\n|location#左肺上叶尖段|\n|size_long_axis_cm#NULL|\n|size_short_axis_cm#NULL|\n|density#实性|\n|shape#小结节样|\n|suspected_diagnosis#陈旧性病变|\n|左侧肺部低密度区 1#NULL|\n|low_density_id1#NULL|\n|location#NULL|\n|size_long_axis_cm#NULL|\n|size_width_axis_cm#NULL|\n|size_height_axis_cm#NULL|\n|shape#NULL|\n|suspected_diagnosis#NULL|\n|side右| |\n|nodules_present#TRUE|\n|nod

In [25]:
# rx答案整理
rx_csv = {}
for idx, rsult_ in enumerate(result_rx):
    print("index:",idx)
    # print(rsult_)
    rx_csv[idx]={}
    rx_csv[idx]["description"] = rsult_["description"]

    # ans_bc = rsult_["ans_baichuan"]
    # output_struct_rx_cp = output_struct_rx.copy()
    # keyword = list(output_struct_rx_cp.keys())
    # ans_bc_values = [v for v in ("报告信息\n"+ans_bc).split("\n") if "#"in v or v in keyword]
    # print("baichuan ans 整理...")
    # rx_csv[idx]["baichuan"] = past_out(ans_bc_values,output_struct_rx_cp)

    
    ans_bc = rsult_["ans_hunyuan"]
    output_struct_rx_cp = output_struct_rx.copy()
    keyword = list(output_struct_rx_cp.keys())
    ans_bc_values = [v for v in ("报告信息\n"+ans_bc).split("\n") if "#"in v or v in keyword]
    print("hunyuan ans 整理...")
    rx_csv[idx]["hunyuan"] = past_out(ans_bc_values,output_struct_rx_cp)

index: 0
hunyuan ans 整理...
报告信息
字段值#报告信息
report_id#1
patient_id#1001
exam_date#2024-12-17
layer_structure#双侧乳腺腺体内部结构稍紊乱
gland_arrangement#回声欠均匀
axillary_findings#NULL
supraclavicular_findings#NULL
CDFI_findings#腺体内未见异常血流信号
乳腺结构是否正常#不正常
左侧乳腺信息#side左
duct_dilation#FALSE
duct_dilation_details#NULL
blood_flow_present#FALSE
blood_flow_details#未见明显血流信号
nodules_present#FALSE
nodules_count#0
右侧乳腺信息#side右
duct_dilation#FALSE
duct_dilation_details#NULL
blood_flow_present#FALSE
blood_flow_details#未见明显血流信号
nodules_present#FALSE
nodules_count#0
结论#conclusion#双侧乳腺增生,符合BI-RADS[2]类
index: 1
hunyuan ans 整理...
报告信息
|report_id|#1|
|patient_id|#NULL|
|exam_date|#NULL|
|layer_structure|#双侧乳腺腺体结构轻度紊乱|
|gland_arrangement|#回声欠均匀|
|axillary_findings|#NULL|
|supraclavicular_findings|#NULL|
|CDFI_findings|#NULL|
|乳腺结构是否正常|#不正常|
|左侧乳腺信息#side左|TRUE|
|duct_dilation_details|#NULL|
|blood_flow_details|#NULL|
|左侧乳腺结节 1#nodule_id1|TRUE|
|boundary|#NULL|
|shape|#NULL|
|blood_flow_details|#NULL|
|右侧乳腺信息#side右|TRUE|
|duct

## 输出整理结果存储

In [33]:
with open("results/outputs_postprocessing/ct_save_v3.json","w")as f:
    json.dump(ct_csv,f,ensure_ascii=False,indent=4)

with open("results/outputs_postprocessing/rx_save_v3.json","w")as f:
    json.dump(rx_csv,f,ensure_ascii=False,indent=4)

## 保存为方便看的csv格式

In [31]:
cache_dict = {}
cache_dict.get("li","")

''

In [26]:
def save_2_df(ct_csv,save_file,axis_=1):
    data_list_ct = []
    for idx,df_data in ct_csv.items():
        description = df_data["description"]
        # baichuan_gen = df_data["baichuan"]
        hunyuan_gen = df_data["hunyuan"]
        data_list_ = []
        dict_cache = {}
        dict_cache["字段"] = description
        dict_cache["百川_结果"] = ""
        dict_cache["混元_结果"] = ""
        data_list_.append(dict_cache)
        for key,value in baichuan_gen.items():
            dict_cache = {}
            dict_cache["字段"] = key
            dict_cache["百川_结果"] = ""
            dict_cache["混元_结果"] = ""
            data_list_.append(dict_cache)
            value_hy = hunyuan_gen.get(key,{})
            for k,v_bc  in value.items():
                dict_cache = {}
                dict_cache["字段"] = k
                dict_cache["百川_结果"] = v_bc
                dict_cache["混元_结果"] = value_hy.get(k,"NULL")
                data_list_.append(dict_cache)
        data_list_ct.append(data_list_)
    df_list = []
    for data_2_df in data_list_ct:
        df_ = pd.DataFrame(data_2_df)
        df_list.append(df_)

    # df_save_ct = pd.concat(df_list,axis=axis_) 
    # save_path = os.path.join("results_v2",save_file)
    # df_save_ct.to_csv(save_path,index=False)
    return df_list


def save_3_df(ct_csv,save_file,axis_=1):
    data_list_ct = []
    for idx,df_data in ct_csv.items():
        description = df_data["description"]
        # baichuan_gen = df_data["baichuan"]
        hunyuan_gen = df_data["hunyuan"]
        data_list_ = []
        dict_cache = {}
        dict_cache["字段"] = description
        # dict_cache["百川_结果"] = ""
        dict_cache["混元_结果"] = ""
        data_list_.append(dict_cache)
        for key,value in hunyuan_gen.items():
            dict_cache = {}
            dict_cache["字段"] = key
            # dict_cache["百川_结果"] = ""
            dict_cache["混元_结果"] = ""
            data_list_.append(dict_cache)
            for k,v_bc  in value.items():
                dict_cache = {}
                dict_cache["字段"] = k
                dict_cache["混元_结果"] = v_bc
                data_list_.append(dict_cache)
        data_list_ct.append(data_list_)
    df_list = []
    for data_2_df in data_list_ct:
        df_ = pd.DataFrame(data_2_df)
        df_list.append(df_)

    # df_save_ct = pd.concat(df_list,axis=axis_) 
    # save_path = os.path.join("results_v2",save_file)
    # df_save_ct.to_csv(save_path,index=False)
    return df_list

In [42]:
for idx,df_data in ct_csv.items():
    key_name = "baichuan"
    print(df_data[key_name])
    print(type(df_data[key_name]))

{'报告信息': {'report_id': '1', 'patient_id': '1001', 'exam_date': '2024-12-17', 'exam_location': '肺部', '肺部结构是否正常': 'NULL'}}
<class 'dict'>
{'报告信息': {'report_id': '1', 'patient_id': '1001', 'exam_date': '2024-12-17', 'exam_location': '肺部', '肺部结构是否正常': '正常'}}
<class 'dict'>
{'报告信息': {'report_id': 'NULL', 'patient_id': 'NULL', 'exam_date': 'NULL', 'exam_location': '肺部', '肺部结构是否正常': '不正常'}, '肺部结构信息': {'layer_structure': '纵隔居中', 'airway_description': '主气管及双侧支气管通畅', 'mediastinal_findings': '纵隔内示多发中小淋巴结影，边界尚清', 'cardiac_large_vessels': '心影血管界面清晰', 'pleural_and_cavity_findings': '双侧胸腔未见明显积液'}, '左侧肺部信息': {'side': '左', 'nodules_present': 'TRUE', 'nodules_count': '1'}, '左侧肺部结节 1': {'nodule_id': '1', 'location': '左肺下叶', 'size_long_axis_cm': 'NULL', 'size_short_axis_cm': 'NULL', 'density': '实性', 'shape': '微小/小', 'suspected_diagnosis': '磨玻璃结节'}, '右侧肺部信息': {'side': '右', 'nodules_present': 'TRUE', 'nodules_count': '3'}, '右侧肺部结节 1': {'nodule_id': '2', 'location': '右肺下叶', 'size_long_axis_cm': '0.3-0.6', 's

In [27]:
df_save_ct_out = save_3_df(ct_csv,"ct影像.csv",0)

In [28]:
df_save_rx_out = save_3_df(rx_csv,"乳腺影像.csv",0)

In [29]:
with pd.ExcelWriter('results/胸部ct_10_v3.xlsx') as writer:
    # 将df1存储在第一个sheet中
    for idx,df_one in enumerate(df_save_ct_out):
        sheet_ = f"case_{idx}"
        df_one.to_excel(writer, sheet_name=sheet_)
    # 将df2存储在第二个sheet中
    # df_save_rx_out.to_excel(writer, sheet_name='乳腺')

In [30]:
with pd.ExcelWriter('results/乳腺_34_v3.xlsx') as writer:
    # 将df1存储在第一个sheet中
    for idx,df_one in enumerate(df_save_rx_out):
        sheet_ = f"case_{idx}"
        df_one.to_excel(writer, sheet_name=sheet_)

# 甲状腺测试

In [176]:
prompt_jia_v2 = """
任务说明：

您是一个医疗数据提取助手，专门用于解析甲状腺超声报告。您的任务是从给定的报告文本中提取以下结构化信息，并将其填写在指定的单一垂直表格中。请确保信息的准确性和完整性，特别是在提取占位病变描述和结论部分。

需要提取的信息类别：

1. 甲状腺结构描述：

   - 内部回声（均匀、不均、欠均匀等）。
   - 包膜（清晰、完整等）。
   - 脏器大小（正常、增大等）。
   - 峡部大小（未见异常、增大等）。
   - 形态（正常、规则等）。
   - 表面（光滑、粗糙）。

2. 占位病变的结构化描述：

   - 类型。
   - TI-RADS分级（0级、1级等）。
   - 位置（甲状腺左侧叶、甲状腺右侧叶等）。
   - 别名。
   - 回声强弱（回声增强、回声减弱等）。
   - 大小（结节超过1cm、结节不超过1cm等）。
   - 性状特征。
   - 数目（单个、多个等）。
   - 轮廓（正常、毛刺状）。
   - 边界（清晰、规整等）。
   - 形态。
   - 横断面状态。

3. CDFI结果：

   - 描述（未见血流信号、血流信号未见明显异常等）。

4. 报告结论的结构化描述：

   - 总结报告中的结论部分，包括任何建议或进一步评估的建议。

表格格式要求：

将所有提取的信息按照以下垂直排列的表格格式填写。每个字段在表格的第一列，相关值在第二列。对于存在多个占位病变的情况，使用编号区分每个占位病变的信息。

字段值

甲状腺信息

内部回声[填写甲状腺内部回声描述，例如“均匀”或“不均匀”]

包膜[填写甲状腺包膜描述，例如“清晰”或“不完整”]

脏器大小[填写甲状腺大小描述，例如“正常”或“弥漫性增大”]

峡部大小[填写峡部大小描述，例如“未见异常”或“弥漫性增大”]

形态[填写甲状腺形态描述，例如“规则”或“不规则”]

表面[填写甲状腺表面描述，例如“光滑”或“粗糙”]

CDFI_findings[填写CDFI结果描述，例如“未见血流信号”或“血流信号未见明显异常”]

占位病变1

lesion_id1

TI-RADS分级[填写TI-RADS分级，例如“TI-RADS 3级”]

位置[填写占位病变位置，例如“甲状腺左侧叶”]

回声强弱[填写占位病变回声强弱，例如“回声增强”或“回声减弱”]

大小[填写占位病变大小，例如“结节超过1cm”]

性状特征[填写占位病变性状特征]

数目[填写占位病变数目，例如“单个”]

轮廓[填写占位病变轮廓，例如“正常”或“毛刺状”]

边界[填写占位病变边界，例如“清晰”或“不清晰”]

形态[填写占位病变形态]

横断面状态[填写占位病变横断面状态]

[根据需要，可以继续添加占位病变2、占位病变3等]

示例输入：
 报告描述：{input}

输出要求：
    1、字段和相关值用#连接，请一定遵守，方便后期处理
    2、无法从示例输入中找到的信息就填NULL
输出示例：
   位置#甲状腺左侧叶
   回声强弱#无回声
   大小#[根据报告内容，若未提及则留空或填写“未提及”]
   性状特征#薄壁囊肿，内部透声好
   数目#单个
   轮廓#[根据报告内容，若未提及则留空或填写“未提及”]
   边界#清晰
   形态#规则
"""

In [170]:
input_text_jia = "甲状腺彩色多普勒超声探查甲状腺形态大小正常,
甲状腺左侧叶探及一大小约2.5*2.5mm的薄壁无回声区,内部透声好,未探及血流信号
甲状腺左侧叶另探及一大小约3.5*2.5mm的低回声结节,边界可见,形态规则,内部回声欠均质,结节内探及血流信号;
甲状腺右侧叶探及多个低回声结节,大者约7*4mm,边界可见,形态规则,内部回声欠均质,结节内探及血流信号
余部甲状腺回声均匀,血流信号未见异常。双侧颈部未探及异常增大淋巴结"
input_text_jia_concl = "甲状腺双侧叶实性结节,TI-RADS-3级"
input_text_jia_concl_2 = "甲状腺左(L)侧叶囊肿,TI-RADS-2级"

In [177]:
prompt_jia_input = prompt_jia_v2.format(input=input_text_jia)
ans_jia = qa_baichuan(prompt_jia_input)
print(ans_jia)

根据提供的示例输入，我将为您提取并整理所需的信息。

```
甲状腺信息#内部回声#均匀
甲状腺信息#包膜#清晰
甲状腺信息#脏器大小#正常
甲状腺信息#峡部大小#未见异常
甲状腺信息#形态#规则
甲状腺信息#表面#光滑
甲状腺信息#CDFI_findings#血流信号未见异常

占位病变1#TI-RADS分级#TI-RADS 2级
占位病变1#位置#甲状腺左侧叶
占位病变1#回声强弱#无回声
占位病变1#大小#2.5*2.5mm
占位病变1#性状特征#薄壁囊肿，内部透声好
占位病变1#数目#单个
占位病变1#轮廓#清晰
占位病变1#边界#清晰
占位病变1#形态#规则
占位病变1#横断面状态#未提及

占位病变2#TI-RADS分级#TI-RADS 3级
占位病变2#位置#甲状腺左侧叶
占位病变2#回声强弱#低回声
占位病变2#大小#3.5*2.5mm
占位病变2#性状特征#边界可见，形态规则，内部回声欠均质
占位病变2#数目#单个
占位病变2#轮廓#清晰
占位病变2#边界#清晰
占位病变2#形态#规则
占位病变2#横断面状态#未提及

占位病变3#TI-RADS分级#TI-RADS 3级
占位病变3#位置#甲状腺右侧叶
占位病变3#回声强弱#低回声
占位病变3#大小#7*4mm
占位病变3#性状特征#边界可见，形态规则，内部回声欠均质
占位病变3#数目#多个
占位病变3#轮廓#清晰
占位病变3#边界#清晰
占位病变3#形态#规则
占位病变3#横断面状态#未提及
```

请注意，TI-RADS分级是根据结节的特征进行推断的，并非报告中直接给出，因此可能存在一定的主观性。其他未提及的信息我填写为"未提及"或者留空。


In [179]:
prompt_ouput = """
任务说明
    您是一个医疗数据提取助手，专门用于解析甲状腺超声报告。我已经从给定的报告文本中提取出了结构化信息，您的任务是围绕结论找到结论相关的结构信息
    报告文本：{input_text}
    结论：{concl}
    报告相关的结构信息：{struct_data}

"""

In [180]:
prompt_jia_input = prompt_ouput.format(input_text=input_text_jia,concl=input_text_jia_concl_2,struct_data=ans_jia)
ans_jia_with_concl = qa_baichuan(prompt_jia_input)
print(ans_jia_with_concl)

根据您的任务说明和提供的报告文本，我已经提取并整理了与结论相关的结构信息。以下是与甲状腺左侧叶囊肿（TI-RADS-2级）相关的结构化信息：

```
甲状腺信息#内部回声#均匀
甲状腺信息#包膜#清晰
甲状腺信息#脏器大小#正常
甲状腺信息#峡部大小#未见异常
甲状腺信息#形态#规则
甲状腺信息#表面#光滑
甲状腺信息#CDFI_findings#血流信号未见异常

占位病变1#TI-RADS分级#TI-RADS 2级
占位病变1#位置#甲状腺左侧叶
占位病变1#回声强弱#无回声
占位病变1#大小#2.5*2.5mm
占位病变1#性状特征#薄壁囊肿，内部透声好
占位病变1#数目#单个
占位病变1#轮廓#清晰
占位病变1#边界#清晰
占位病变1#形态#规则
占位病变1#横断面状态#未提及
```

这些信息详细描述了甲状腺左侧叶囊肿的特征，包括其位置、大小、形态、边界、内部回声等，以及TI-RADS分级为2级。希望这些信息能够满足您的需求。如果您还有其他问题或需要进一步的帮助，请随时告诉我。


In [185]:
print("报告内容：",input_text_jia)
print("结论：",input_text_jia_concl_2)
print("结构化信息：",ans_jia_with_concl)

报告内容： 甲状腺彩色多普勒超声探查甲状腺形态大小正常,甲状腺左侧叶探及一大小约2.5*2.5mm的薄壁无回声区,内部透声好,未探及血流信号甲状腺左侧叶另探及一大小约3.5*2.5mm的低回声结节,边界可见,形态规则,内部回声欠均质,结节内探及血流信号;甲状腺右侧叶探及多个低回声结节,大者约7*4mm,边界可见,形态规则,内部回声欠均质,结节内探及血流信号余部甲状腺回声均匀,血流信号未见异常。双侧颈部未探及异常增大淋巴结
结论： 甲状腺左(L)侧叶囊肿,TI-RADS-2级
结构化信息： 根据您的任务说明和提供的报告文本，我已经提取并整理了与结论相关的结构信息如下：

```
甲状腺信息#内部回声#均匀
甲状腺信息#包膜#清晰
甲状腺信息#脏器大小#正常
甲状腺信息#峡部大小#未见异常
甲状腺信息#形态#规则
甲状腺信息#表面#光滑
甲状腺信息#CDFI_findings#血流信号未见异常

占位病变1#TI-RADS分级#TI-RADS 2级
占位病变1#位置#甲状腺左侧叶
占位病变1#回声强弱#无回声
占位病变1#大小#2.5*2.5mm
占位病变1#性状特征#薄壁囊肿，内部透声好
占位病变1#数目#单个
占位病变1#轮廓#清晰
占位病变1#边界#清晰
占位病变1#形态#规则
占位病变1#横断面状态#未提及

占位病变2#TI-RADS分级#TI-RADS 3级
占位病变2#位置#甲状腺左侧叶
占位病变2#回声强弱#低回声
占位病变2#大小#3.5*2.5mm
占位病变2#性状特征#边界可见，形态规则，内部回声欠均质
占位病变2#数目#单个
占位病变2#轮廓#清晰
占位病变2#边界#清晰
占位病变2#形态#规则
占位病变2#横断面状态#未提及

占位病变3#TI-RADS分级#TI-RADS 3级
占位病变3#位置#甲状腺右侧叶
占位病变3#回声强弱#低回声
占位病变3#大小#7*4mm
占位病变3#性状特征#边界可见，形态规则，内部回声欠均质
占位病变3#数目#多个
占位病变3#轮廓#清晰
占位病变3#边界#清晰
占位病变3#形态#规则
占位病变3#横断面状态#未提及
```

请注意，TI-RADS分级是根据结节的特征进行推断的，并非报告中直接给出，因此可能存在一定的主观性。其他未提及的信息我填写为"未提及"或者留空。


In [183]:
print(input_text_jia_concl)

甲状腺双侧叶实性结节,TI-RADS-3级


In [184]:
prompt_jia_input = prompt_ouput.format(input_text=input_text_jia,concl=input_text_jia_concl,struct_data=ans_jia)
ans_jia_with_concl = qa_baichuan(prompt_jia_input)
print(ans_jia_with_concl)

根据您的任务说明和提供的报告文本，我已经提取并整理了与结论相关的结构信息如下：

```
甲状腺信息#内部回声#均匀
甲状腺信息#包膜#清晰
甲状腺信息#脏器大小#正常
甲状腺信息#峡部大小#未见异常
甲状腺信息#形态#规则
甲状腺信息#表面#光滑
甲状腺信息#CDFI_findings#血流信号未见异常

占位病变1#TI-RADS分级#TI-RADS 2级
占位病变1#位置#甲状腺左侧叶
占位病变1#回声强弱#无回声
占位病变1#大小#2.5*2.5mm
占位病变1#性状特征#薄壁囊肿，内部透声好
占位病变1#数目#单个
占位病变1#轮廓#清晰
占位病变1#边界#清晰
占位病变1#形态#规则
占位病变1#横断面状态#未提及

占位病变2#TI-RADS分级#TI-RADS 3级
占位病变2#位置#甲状腺左侧叶
占位病变2#回声强弱#低回声
占位病变2#大小#3.5*2.5mm
占位病变2#性状特征#边界可见，形态规则，内部回声欠均质
占位病变2#数目#单个
占位病变2#轮廓#清晰
占位病变2#边界#清晰
占位病变2#形态#规则
占位病变2#横断面状态#未提及

占位病变3#TI-RADS分级#TI-RADS 3级
占位病变3#位置#甲状腺右侧叶
占位病变3#回声强弱#低回声
占位病变3#大小#7*4mm
占位病变3#性状特征#边界可见，形态规则，内部回声欠均质
占位病变3#数目#多个
占位病变3#轮廓#清晰
占位病变3#边界#清晰
占位病变3#形态#规则
占位病变3#横断面状态#未提及
```

请注意，TI-RADS分级是根据结节的特征进行推断的，并非报告中直接给出，因此可能存在一定的主观性。其他未提及的信息我填写为"未提及"或者留空。
